# MILK10k Bronze ingestion

This notebook expects the MILK10k Bronze tables/folders to already exist and the source archive to already be uploaded to landing.

Most of the archive-handling logic lives in `data_platform.files`/`data_platform.spark_io` as shared functions (assuming archive-based ingestion, which is what MILK10k uses too — a single archive, no train/test split, unlike ISIC 2019's two archives). The one genuinely MILK10k-specific step is the Bronze source metadata schema in "Write Bronze Tables", which stays inline since MILK10k's raw metadata columns don't exactly match ISIC 2019's (fewer diagnosis/anatom_site levels, no patient_id, no clin_size_long_diam_mm/dermoscopic_type/family_hx_mm/personal_hx_mm, plus a new image_manipulation column).

Run `05_setup_tables_and_folders.ipynb` before this notebook.


In [ ]:
%run ../_setup_env


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from pyspark.sql import functions as F

from data_platform.files import check_archives_exist, stage_archives_and_extract_metadata
from data_platform.dataset_layout import build_layout, load_dataset_config, load_yaml_config, resolve_archive_paths
from data_platform.spark_io import (
    load_combined_metadata,
    print_bronze_outputs,
    write_image_index_table,
    write_ingestion_run,
)

import tempfile


DATASET_CONFIG = load_dataset_config(CONFIG_ROOT / "bronze" / "datasets" / "milk10k.yaml")
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
DATASET_CONFIG["landing_root"] = STORAGE_CONFIG["landing_root"]
DATASET_CONFIG["storage_root"] = STORAGE_CONFIG["storage_root"]
layout = build_layout(DATASET_CONFIG)
landing_paths = layout["landing_paths"]
bronze_paths = layout["bronze_paths"]
bronze_tables = layout["bronze_tables"]

ARCHIVES = resolve_archive_paths(DATASET_CONFIG["archives"], landing_paths, bronze_paths)
LOCAL_ARCHIVE_STAGE_ROOT = Path(tempfile.gettempdir()) / DATASET_CONFIG["dataset_key"] / "archive_stage"
OVERWRITE_LOCAL_STAGE = False

for archive in ARCHIVES:
    print(f"{archive['source_split']} archive landing path: {archive['archive_dbfs_path']}")
    print(f"{archive['source_split']} Bronze image index table: {bronze_tables['image_index']}")
    print(f"{archive['source_split']} Bronze metadata path: {archive['metadata_target_path']}")


## Preflight

`check_archives_exist` verifies that the required archive already exists in landing before doing any extraction or table writes. Assumes archive-based ingestion, shared via `data_platform.files`.


In [ ]:
check_archives_exist(ARCHIVES)


## Stage Archive and Metadata

`stage_archives_and_extract_metadata` copies the source archive once from the landing Volume to local disk, then extracts only the embedded metadata CSV into Bronze metadata. Image extraction and byte reads happen from local disk in the next step. Assumes archive-based ingestion, shared via `data_platform.files`.


In [ ]:
stage_archives_and_extract_metadata(ARCHIVES, LOCAL_ARCHIVE_STAGE_ROOT, OVERWRITE_LOCAL_STAGE)

display(dbutils.fs.ls(bronze_paths["metadata"]))


## Build And Write Image Index Table

`write_image_index_table` streams the locally staged archive to compute each image's checksum/byte_length and writes index rows (no image bytes) to one Delta table in bounded batches — it never creates one object per JPEG in the Volume, and never persists image bytes anywhere at all (see `docs/decisions/006-stream-archives-no-blob-storage.md`). Assumes archive-based ingestion, shared via `data_platform.spark_io`.


In [ ]:
image_manifest_df = write_image_index_table(spark, ARCHIVES, bronze_tables["image_index"])


## Write Bronze Tables

`load_combined_metadata` combines each archive's raw metadata (shared, assumes one CSV per archive — here there's only one). The metadata schema/select and its `MERGE INTO` below are MILK10k's own Bronze contract — genuinely dataset-specific, stays inline. `write_ingestion_run` then appends the run record to the shared, ingestion-source-agnostic `bronze.ingestion_runs` table.


In [ ]:
raw_metadata_df = load_combined_metadata(spark, ARCHIVES)

ingestion_run_id = f"bronze-{uuid4().hex[:12]}"
started_at = datetime.now(timezone.utc)

# MILK10k's actual Bronze source metadata schema — dataset-specific by design, unlike
# everything else in this notebook (see docs/data_contract.md, AGENT.md's "Table strategy").
# Fewer diagnosis/anatom_site levels than ISIC 2019, no patient_id/clin_size_long_diam_mm/
# dermoscopic_type/family_hx_mm/personal_hx_mm, plus a new image_manipulation column.
source_metadata_df = (
    raw_metadata_df
    .withColumn("image_id", F.col("isic_id").cast("string"))
    .join(image_manifest_df, on=["image_id", "source_split"], how="left")
    .select(
        F.col("image_id"),
        F.col("source_split"),
        F.col("source_uri"),
        F.col("attribution"),
        F.col("copyright_license"),
        F.col("age_approx"),
        F.col("anatom_site_1"),
        F.col("anatom_site_2"),
        F.col("anatom_site_special"),
        F.col("concomitant_biopsy"),
        F.col("diagnosis_1"),
        F.col("diagnosis_2"),
        F.col("diagnosis_3"),
        F.col("diagnosis_4"),
        F.col("diagnosis_confirm_type"),
        F.col("image_manipulation"),
        F.col("image_type"),
        F.col("lesion_id"),
        F.col("melanocytic"),
        F.col("sex"),
        F.col("source_checksum"),
    )
    .where(F.col("image_id").isNotNull())
    .withColumn("ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("ingested_at", F.lit(started_at))
)

source_metadata_df.createOrReplaceTempView("staged_milk10k_source_metadata")
spark.sql(
    f"""
    MERGE INTO {bronze_tables['source_metadata']} AS target
    USING staged_milk10k_source_metadata AS source
    ON target.image_id = source.image_id AND target.source_split = source.source_split
    WHEN MATCHED THEN UPDATE SET
      target.source_split = source.source_split,
      target.source_uri = source.source_uri,
      target.attribution = source.attribution,
      target.copyright_license = source.copyright_license,
      target.age_approx = source.age_approx,
      target.anatom_site_1 = source.anatom_site_1,
      target.anatom_site_2 = source.anatom_site_2,
      target.anatom_site_special = source.anatom_site_special,
      target.concomitant_biopsy = source.concomitant_biopsy,
      target.diagnosis_1 = source.diagnosis_1,
      target.diagnosis_2 = source.diagnosis_2,
      target.diagnosis_3 = source.diagnosis_3,
      target.diagnosis_4 = source.diagnosis_4,
      target.diagnosis_confirm_type = source.diagnosis_confirm_type,
      target.image_manipulation = source.image_manipulation,
      target.image_type = source.image_type,
      target.lesion_id = source.lesion_id,
      target.melanocytic = source.melanocytic,
      target.sex = source.sex,
      target.source_checksum = source.source_checksum,
      target.ingestion_run_id = source.ingestion_run_id,
      target.ingested_at = source.ingested_at
    WHEN NOT MATCHED THEN INSERT *
    """
)

records_written = source_metadata_df.count()
finished_at = datetime.now(timezone.utc)
write_ingestion_run(
    spark,
    bronze_tables,
    ingestion_run_id,
    DATASET_CONFIG["dataset_name"],
    DATASET_CONFIG["source_version"],
    started_at,
    finished_at,
    records_seen=records_written,
    records_written=records_written,
)

print(f"Ingestion run id: {ingestion_run_id}")
print(f"Records written: {records_written}")


## Review Outputs

`print_bronze_outputs` shows the landed archive, Bronze file paths, and the most recent Bronze table contents for inspection. Fully dataset-agnostic, shared via `data_platform.spark_io`.


In [ ]:
print_bronze_outputs(spark, dbutils, display, landing_paths, bronze_paths, bronze_tables, DATASET_CONFIG["dataset_name"])
